# Multi-Fidelity qMFKG Campaign Tutorial

This notebook demonstrates BO Forge's conservative single-objective multi-fidelity workflow with one continuous fidelity variable and qMFKG suggestions.

The CSV log remains the source of truth. Suggestions are dry runs until they are explicitly appended, and lower-fidelity observations are real measured objective values at the requested fidelity.

In [ ]:
from pathlib import Path
import math
import os
import shutil
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

mpl_cache = PROJECT_ROOT / ".matplotlib-cache"
mpl_cache.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from bo_forge import CampaignSession

CONFIG_PATH = PROJECT_ROOT / "configs" / "15_multi_fidelity_qmfkg.yaml"
SEED_LOG_PATH = PROJECT_ROOT / "examples" / "15_multi_fidelity_qmfkg_campaign_log.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)
WORKING_LOG_PATH = PROJECT_ROOT / "examples" / "15_multi_fidelity_qmfkg_working_log.csv"
LATEST_SUGGESTIONS_PATH = PROJECT_ROOT / "examples" / "15_multi_fidelity_qmfkg_latest_suggestions.csv"
REPORT_PATH = REPORTS_DIR / "15_multi_fidelity_qmfkg_report.txt"
PROGRESS_PATH = REPORTS_DIR / "15_multi_fidelity_qmfkg_progress.png"
DIAGNOSTICS_PATH = REPORTS_DIR / "15_multi_fidelity_qmfkg_diagnostics.png"
FIDELITY_DIAGNOSTICS_PATH = REPORTS_DIR / "15_multi_fidelity_qmfkg_fidelity_diagnostics.png"
TARGET_OBSERVED_ROWS = 15

shutil.copyfile(SEED_LOG_PATH, WORKING_LOG_PATH)
campaign = CampaignSession.from_files(CONFIG_PATH, WORKING_LOG_PATH)

## Validate and inspect the campaign

`summary()` gives the campaign-level view. `fidelity_summary()` reports observed fidelity coverage, target-fidelity observations, pending qMFKG suggestions, and direction-aware best rows.

In [ ]:
campaign.validate()
campaign.summary()

In [ ]:
campaign.fidelity_summary()

In [ ]:
campaign.next_action()

## Deterministic local evaluator

The tutorial uses a small deterministic function so the notebook can run end to end. In a real campaign this step would be replaced by the experiment measured at the suggested fidelity.

In [ ]:
def simulate_activity(row):
    loading = float(row["catalyst_loading"])
    fidelity = float(row["fidelity"])
    target_activity = 1.08 + 1.35 * loading - 2.0 * (loading - 0.38) ** 2
    fidelity_bias = -0.80 * (1.0 - fidelity)
    smooth_variation = 0.06 * math.sin(8.0 * loading + 3.0 * fidelity)
    return round(target_activity + fidelity_bias + smooth_variation, 6)

## Sequential qMFKG loop

Once the initial design is complete, v1.4 multi-fidelity qMFKG suggestions are generated one at a time. The loop below writes each latest dry-run suggestion to an ignored CSV, appends explicitly, simulates the observed value, and records it with `mark_observed()`.

In [ ]:
records = []
while len(campaign.observed_data()) < TARGET_OBSERVED_ROWS:
    suggestions = campaign.suggest_next(batch_size=1)
    suggestions.to_csv(LATEST_SUGGESTIONS_PATH, index=False)
    campaign.append_suggestions(suggestions)
    for row_id in suggestions["row_id"]:
        row = campaign.df.loc[campaign.df["row_id"] == row_id].iloc[0]
        observed_value = simulate_activity(row)
        campaign.mark_observed(row_id=row_id, objective_value=observed_value)
        records.append(
            {
                "row_id": row_id,
                "catalyst_loading": float(row["catalyst_loading"]),
                "fidelity": float(row["fidelity"]),
                "activity": observed_value,
            }
        )

campaign.summary()

In [ ]:
campaign.fidelity_summary()

## Export report and diagnostics

The report includes a fidelity summary. The diagnostics are read-only and use the observed CSV rows rather than fitting another model.

In [ ]:
campaign.export_report(REPORT_PATH)
_progress_fig, _progress_ax = campaign.plot_progress(save_path=PROGRESS_PATH)
_diagnostics_fig, _diagnostics_ax = campaign.plot_diagnostics(save_path=DIAGNOSTICS_PATH)
_fidelity_fig, _fidelity_axes = campaign.plot_fidelity_diagnostics(
    save_path=FIDELITY_DIAGNOSTICS_PATH
)
REPORT_PATH, PROGRESS_PATH, DIAGNOSTICS_PATH, FIDELITY_DIAGNOSTICS_PATH

## Compact CLI demo

The same read-only fidelity inspection is available through `python -m bo_forge`.

In [ ]:
completed = subprocess.run(
    [
        sys.executable,
        "-m",
        "bo_forge",
        "fidelity-summary",
        "--config",
        str(CONFIG_PATH),
        "--log",
        str(WORKING_LOG_PATH),
    ],
    check=True,
    capture_output=True,
    cwd=PROJECT_ROOT,
    text=True,
)
print(completed.stdout)

## Limitations

- v1.4 multi-fidelity support is single-objective only.
- The fidelity variable must be continuous, and all design variables in this workflow are continuous.
- qMFKG model-based suggestions are sequential with `batch_size=1`.
- Multi-objective, structured, cost-aware, replicate-aware, discrete, and categorical multi-fidelity workflows remain deferred.
- The notebook demonstrates backend/session/CLI behavior; it is not the campaign engine.